# IKEA RFM Customer Segmentation
### Python • Pandas • Customer Analytics

**Project objective**  
Use transaction-level retail data to calculate **Recency, Frequency, and Monetary (RFM)** metrics, assign customer segments, flag potentially churned customers, and export a clean dataset for Power BI.

**Workflow**
1. Load and validate the cleaned transaction dataset.
2. Prepare transaction dates and calculate revenue.
3. Calculate RFM metrics at customer level.
4. Score customers and assign business segments.
5. Add customer metrics back to transaction records.
6. Export the result for dashboarding.

> **Before running:** Keep `RetailCleaned.csv` in the same folder as this notebook. The notebook expects nine columns in this order: Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer_ID, Country, SourceSheet. Confirm that this file contains the intended IKEA project data before publishing.

## 1. Import libraries and configure paths

This notebook uses pandas for data preparation and Excel export. If Excel export raises a missing-engine error, install `openpyxl` in your Python environment.

In [ ]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("RetailCleaned.csv")
OUTPUT_FILE = Path("EcommerceDashboardData.xlsx")

EXPECTED_COLUMNS = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer_ID", "Country", "SourceSheet"
]

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {INPUT_FILE.resolve()}. "
        "Place RetailCleaned.csv in the same folder as this notebook."
    )

print("Input file:", INPUT_FILE.resolve())
print("Output file:", OUTPUT_FILE.resolve())

## 2. Load and inspect the data

The source file is loaded without a header, then the expected column names are assigned. Dates and numeric fields are converted to suitable data types.

In [ ]:
df = pd.read_csv(INPUT_FILE, header=None, low_memory=False)

if df.shape[1] != len(EXPECTED_COLUMNS):
    raise ValueError(
        f"Expected {len(EXPECTED_COLUMNS)} columns, but found {df.shape[1]}. "
        "Check the source file and column order."
    )

df.columns = EXPECTED_COLUMNS

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Customer_ID"] = pd.to_numeric(df["Customer_ID"], errors="coerce")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

print("\nData types and non-null counts:")
df.info()

## 3. Check data quality

Review missing values, invalid dates, and duplicate rows before calculating customer metrics. This section reports issues; it does not silently delete records from the source DataFrame.

In [ ]:
quality_report = pd.DataFrame({
    "Missing values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
})
display(quality_report)

print("Exact duplicate rows:", f"{df.duplicated().sum():,}")
print("Invalid invoice dates:", f"{df['InvoiceDate'].isna().sum():,}")
print("Rows with non-positive quantity:", f"{(df['Quantity'].fillna(0) <= 0).sum():,}")
print("Rows with negative price:", f"{(df['Price'].fillna(0) < 0).sum():,}")

## 4. Prepare transaction data and calculate revenue

For RFM calculations, this analysis uses records with a known customer ID and positive quantity. Rows with missing transaction dates or prices are excluded from the calculation dataset.

In [ ]:
transactions = df.loc[
    df["Customer_ID"].notna()
    & df["Quantity"].gt(0)
    & df["InvoiceDate"].notna()
    & df["Price"].notna()
].copy()

transactions["Revenue"] = transactions["Quantity"] * transactions["Price"]
transactions["InvoiceMonth"] = (
    transactions["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
)

print(f"Eligible transaction rows: {len(transactions):,}")
print(f"Unique customers: {transactions['Customer_ID'].nunique():,}")
print(f"Total revenue in eligible records: {transactions['Revenue'].sum():,.2f}")

display(transactions.head())

## 5. Calculate RFM metrics

- **Recency:** Number of days since the customer's most recent purchase.
- **Frequency:** Number of distinct invoices/orders for the customer.
- **Monetary:** Sum of revenue attributed to the customer.

The snapshot date is set to **10 December 2011**, matching the date used in the original analysis. If your dataset covers a different period, update this date to an appropriate reporting date.

In [ ]:
SNAPSHOT_DATE = pd.Timestamp("2011-12-10")

rfm = (
    transactions.groupby("Customer_ID")
    .agg(
        Recency=("InvoiceDate", lambda dates: (SNAPSHOT_DATE - dates.max()).days),
        Frequency=("Invoice", "nunique"),
        Monetary=("Revenue", "sum")
    )
    .reset_index()
)

display(rfm.head())
print(f"Customers scored: {len(rfm):,}")

## 6. Score customers and assign segments

Each RFM metric is divided into five ranked groups. Higher recency scores mean a more recent purchase; higher frequency and monetary scores mean more orders and more spend. Ranking before binning helps handle repeated values in the data.

In [ ]:
def quintile_score(series, labels):
    ranked = series.rank(method="first")
    return pd.qcut(ranked, q=5, labels=labels).astype(int)

rfm["R_Score"] = quintile_score(rfm["Recency"], [5, 4, 3, 2, 1])
rfm["F_Score"] = quintile_score(rfm["Frequency"], [1, 2, 3, 4, 5])
rfm["M_Score"] = quintile_score(rfm["Monetary"], [1, 2, 3, 4, 5])

def segment_customer(row):
    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "Champions"
    elif row["R_Score"] >= 3 and row["F_Score"] >= 3:
        return "Loyal"
    elif row["R_Score"] <= 2 and row["F_Score"] >= 3:
        return "At Risk"
    elif row["R_Score"] >= 4 and row["F_Score"] <= 2:
        return "New"
    elif row["R_Score"] == 3 and row["F_Score"] <= 2:
        return "Needs Attention"
    else:
        return "Lost"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

# A simple rule-based flag: customers whose last purchase is over 90 days before the snapshot date.
rfm["IsChurned"] = rfm["Recency"] > 90

display(rfm.head())

## 7. Review customer segments

Use these summaries to check the customer mix and revenue contribution before building the Power BI report.

In [ ]:
segment_summary = (
    rfm.groupby("Segment", observed=True)
    .agg(
        Customers=("Customer_ID", "nunique"),
        Revenue=("Monetary", "sum"),
        Avg_Recency_Days=("Recency", "mean"),
        Avg_Orders=("Frequency", "mean")
    )
    .sort_values("Revenue", ascending=False)
    .reset_index()
)

segment_summary["Revenue"] = segment_summary["Revenue"].round(2)
segment_summary["Avg_Recency_Days"] = segment_summary["Avg_Recency_Days"].round(1)
segment_summary["Avg_Orders"] = segment_summary["Avg_Orders"].round(1)

display(segment_summary)

print("Potentially churned customers:", f"{rfm['IsChurned'].sum():,}")
print("Total customers:", f"{rfm['Customer_ID'].nunique():,}")

## 8. Create the Power BI output file

The customer-level RFM fields are merged onto each eligible transaction. The resulting Excel file can be imported into Power BI for customer-segment, revenue, order, and churn analysis.

In [ ]:
rfm_columns = [
    "Customer_ID", "Recency", "Frequency", "Monetary",
    "R_Score", "F_Score", "M_Score", "Segment", "IsChurned"
]

dashboard_data = transactions.merge(
    rfm[rfm_columns],
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    dashboard_data.to_excel(writer, sheet_name="Transaction Data", index=False)
    rfm.to_excel(writer, sheet_name="Customer RFM", index=False)
    segment_summary.to_excel(writer, sheet_name="Segment Summary", index=False)

print(f"Export complete: {OUTPUT_FILE.resolve()}")
print(f"Transaction Data rows: {len(dashboard_data):,}")
print(f"Customer RFM rows: {len(rfm):,}")

## 9. Business interpretation notes

Use actual output tables and Power BI visuals to complete these observations; do not assume the results before running the notebook.

- Which segment has the most customers?
- Which segment contributes the highest total monetary value?
- How many customers meet the rule-based churn flag (`Recency > 90`)?
- Which segments may need retention campaigns?
- Are there data-quality limitations that could affect the analysis?

**Important limitation:** `IsChurned` is a simple rule-based indicator, not a confirmed prediction of future churn. The RFM snapshot date and 90-day threshold should be justified for the dataset and business context.